In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# อสมการ CHSH

*ประมาณการการใช้งาน: สองนาทีบนโปรเซสเซอร์ Heron r2 (หมายเหตุ: นี่เป็นเพียงการประมาณการ ระยะเวลาจริงอาจแตกต่างกัน)*
## พื้นหลัง
ในบทแนะนำนี้ คุณจะรันการทดลองบนคอมพิวเตอร์ควอนตัมเพื่อแสดงให้เห็นการละเมิดอสมการ CHSH ด้วย Estimator primitive

อสมการ CHSH ซึ่งตั้งชื่อตามผู้เขียน Clauser, Horne, Shimony และ Holt ถูกใช้เพื่อพิสูจน์ทฤษฎีบทของเบลล์เชิงทดลอง (ค.ศ. 1969) ทฤษฎีบทนี้ยืนยันว่าทฤษฎีตัวแปรซ่อนเร้นเฉพาะที่ไม่สามารถอธิบายผลที่ตามมาบางประการของการพัวพันในกลศาสตร์ควอนตัมได้ การละเมิดอสมการ CHSH ถูกใช้เพื่อแสดงว่ากลศาสตร์ควอนตัมนั้นไม่สอดคล้องกับทฤษฎีตัวแปรซ่อนเร้นเฉพาะที่ นี่เป็นการทดลองที่สำคัญในการทำความเข้าใจรากฐานของกลศาสตร์ควอนตัม

รางวัลโนเบลสาขาฟิสิกส์ปี 2022 มอบให้แก่ Alain Aspect, John Clauser และ Anton Zeilinger ส่วนหนึ่งเนื่องจากผลงานบุกเบิกของพวกเขาในสาขาวิทยาศาสตร์สารสนเทศควอนตัม และโดยเฉพาะอย่างยิ่งสำหรับการทดลองด้วยโฟตอนที่พัวพันกันซึ่งแสดงให้เห็นการละเมิดอสมการของเบลล์
## ข้อกำหนดเบื้องต้น
ก่อนเริ่มบทแนะนำนี้ ตรวจสอบให้แน่ใจว่าคุณได้ติดตั้งสิ่งต่อไปนี้แล้ว:

* Qiskit SDK v1.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 หรือใหม่กว่า
## ตั้งค่า

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

## ขั้นตอนที่ 1: แปลงอินพุตแบบคลาสสิกเป็นปัญหาควอนตัม
สำหรับการทดลองนี้ เราจะสร้างคู่ที่พัวพันกันซึ่งเราวัด Qubit แต่ละตัวบนสองฐานที่แตกต่างกัน เราจะกำหนดชื่อฐานสำหรับ Qubit แรกว่า $A$ และ $a$ และฐานสำหรับ Qubit ที่สองว่า $B$ และ $b$ ซึ่งช่วยให้เราคำนวณปริมาณ CHSH $S_1$ ได้:

$$
S_1 = A(B-b) + a(B+b).
$$

Observable แต่ละตัวมีค่าเป็น $+1$ หรือ $-1$ เห็นได้ชัดว่าหนึ่งในเทอม $B\pm b$ ต้องเป็น $0$ และอีกเทอมต้องเป็น $\pm 2$ ดังนั้น $S_1 = \pm 2$ ค่าเฉลี่ยของ $S_1$ ต้องเป็นไปตามอสมการ:

$$
|\langle S_1 \rangle|\leq 2.
$$

การกระจาย $S_1$ ในรูปของ $A$, $a$, $B$, และ $b$ ให้ผลลัพธ์:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

คุณสามารถกำหนดปริมาณ CHSH อีกค่าหนึ่งคือ $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

ซึ่งนำไปสู่อสมการอีกข้อหนึ่ง:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

หากกลศาสตร์ควอนตัมสามารถอธิบายได้ด้วยทฤษฎีตัวแปรซ่อนเร้นเฉพาะที่ อสมการข้างต้นจะต้องเป็นจริง อย่างไรก็ตาม ดังที่แสดงให้เห็นในบทแนะนำนี้ อสมการเหล่านี้สามารถถูกละเมิดได้ในคอมพิวเตอร์ควอนตัม ดังนั้นกลศาสตร์ควอนตัมจึงไม่สอดคล้องกับทฤษฎีตัวแปรซ่อนเร้นเฉพาะที่
หากต้องการศึกษาทฤษฎีเพิ่มเติม ลองสำรวจ [Entanglement in Action](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game) กับ John Watrous
คุณจะสร้างคู่ที่พัวพันกันระหว่าง Qubit สองตัวในคอมพิวเตอร์ควอนตัมโดยการสร้างสถานะเบลล์ $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$ การใช้ Estimator primitive ช่วยให้คุณได้ค่าความคาดหวังที่ต้องการโดยตรง ($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$ และ $\langle ab \rangle$) เพื่อคำนวณค่าความคาดหวังของปริมาณ CHSH ทั้งสอง $\langle S_1\rangle$ และ $\langle S_2\rangle$ ก่อนการแนะนำ Estimator primitive คุณต้องสร้างค่าความคาดหวังจากผลลัพธ์การวัด

คุณจะวัด Qubit ที่สองในฐาน $Z$ และ $X$ ส่วน Qubit แรกจะวัดในฐานตั้งฉากเช่นกัน แต่มีมุมเทียบกับ Qubit ที่สอง ซึ่งเราจะกวาดระหว่าง $0$ ถึง $2\pi$ ดังที่คุณจะเห็น Estimator primitive ทำให้การรัน Circuit ที่มีพารามิเตอร์เป็นเรื่องง่ายมาก แทนที่จะสร้าง CHSH Circuit หลายตัว คุณต้องสร้างเพียง *หนึ่ง* CHSH Circuit ที่มีพารามิเตอร์ระบุมุมการวัดและชุดค่าเฟสสำหรับพารามิเตอร์นั้น

สุดท้าย คุณจะวิเคราะห์ผลลัพธ์และพล็อตเทียบกับมุมการวัด คุณจะเห็นว่าในช่วงมุมการวัดบางช่วง ค่าความคาดหวังของปริมาณ CHSH $|\langle S_1\rangle| > 2$ หรือ $|\langle S_2\rangle| > 2$ ซึ่งแสดงให้เห็นการละเมิดอสมการ CHSH

In [2]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_kingston'

### Create a parameterized CHSH circuit

First, we write the circuit with the parameter $\theta$, which we call `theta`. The [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) can enormously simplify circuit building and output analysis by directly providing expectation values of observables. Many problems of interest, especially for near-term applications on noisy systems, can be formulated in terms of expectation values. `Estimator` (V2) primitive can automatically change measurement basis based on the supplied observable.

In [3]:
theta = Parameter("$\\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif" alt="Output of the previous code cell" />

### สร้าง CHSH Circuit ที่มีพารามิเตอร์
ก่อนอื่น เราจะเขียน Circuit ที่มีพารามิเตอร์ $\theta$ ซึ่งเราเรียกว่า `theta` [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) สามารถทำให้การสร้าง Circuit และการวิเคราะห์ผลลัพธ์ง่ายขึ้นอย่างมากโดยการให้ค่าความคาดหวังของ Observable โดยตรง ปัญหาหลายอย่างที่น่าสนใจ โดยเฉพาะสำหรับการประยุกต์ใช้งานระยะใกล้บนระบบที่มีสัญญาณรบกวน สามารถกำหนดในรูปของค่าความคาดหวัง `Estimator` (V2) primitive สามารถเปลี่ยนฐานการวัดโดยอัตโนมัติตาม Observable ที่ให้มา

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as list of lists in order to work
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### สร้างรายการค่าเฟสที่จะกำหนดภายหลัง
หลังจากสร้าง CHSH Circuit ที่มีพารามิเตอร์แล้ว คุณจะสร้างรายการค่าเฟสที่จะกำหนดให้กับ Circuit ในขั้นตอนถัดไป คุณสามารถใช้โค้ดต่อไปนี้เพื่อสร้างรายการค่าเฟส 21 ค่าในช่วง $0$ ถึง $2 \pi$ ด้วยระยะห่างเท่ากัน คือ $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$

In [5]:
# <CHSH1> = <AB> - <Ab> + <aB> + <ab> -> <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <CHSH2> = <AB> + <Ab> - <aB> + <ab> -> <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### Observable
ตอนนี้เราต้องการ Observable เพื่อคำนวณค่าความคาดหวัง ในกรณีของเรา เราดูที่ฐานตั้งฉากสำหรับ Qubit แต่ละตัว โดยให้การหมุน $Y$ ที่มีพารามิเตอร์สำหรับ Qubit แรกกวาดฐานการวัดแทบต่อเนื่องเทียบกับฐานของ Qubit ที่สอง ดังนั้นเราจะเลือก Observable $ZZ$, $ZX$, $XZ$ และ $XX$

In [6]:
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif" alt="Output of the previous code cell" />

## ขั้นตอนที่ 2: ปรับแต่งปัญหาสำหรับการรันบนฮาร์ดแวร์ควอนตัม

เพื่อลดเวลาการรันงานทั้งหมด V2 primitives รับเฉพาะ Circuit และ Observable ที่สอดคล้องกับคำสั่งและการเชื่อมต่อที่รองรับโดยระบบเป้าหมาย (เรียกว่า ISA circuits และ observables)

### ISA Circuit

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### ISA Observables

ในทำนองเดียวกัน เราต้องแปลง Observable เพื่อให้ใช้งานได้กับ Backend ก่อนรันงานด้วย [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) เราสามารถทำการแปลงนี้ได้โดยใช้เมธอด `apply_layout` ของออบเจกต์ `SparsePauliOp`

In [8]:
# To run on a local simulator:
# Use the StatevectorEstimator from qiskit.primitives instead.

estimator = Estimator(mode=backend)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA Observables
    individual_phases,  # Parameter values
)

job_result = estimator.run(pubs=[pub]).result()

## ขั้นตอนที่ 3: รันด้วย Qiskit primitives
เพื่อรันการทดลองทั้งหมดในการเรียกครั้งเดียวด้วย [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2)
เราสามารถสร้าง [Qiskit Runtime `Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) primitive เพื่อคำนวณค่าความคาดหวังของเรา เมธอด `EstimatorV2.run()` รับ iterable ของ `primitive unified blocs (PUBs)` แต่ละ PUB เป็น iterable ในรูปแบบ `(circuit, observables, parameter_values: Optional, precision: Optional)`

In [9]:
chsh1_est = job_result[0].data.evs[0]
chsh2_est = job_result[0].data.evs[1]

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))

# results from hardware
ax.plot(phases / np.pi, chsh1_est, "o-", label="CHSH1", zorder=3)
ax.plot(phases / np.pi, chsh2_est, "o-", label="CHSH2", zorder=3)

# classical bound +-2
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")

# quantum bound, +-2√2
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

# set x tick labels to the unit of pi
ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

# set labels, and legend
plt.xlabel("Theta")
plt.ylabel("CHSH witness")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/f6267448-0.avif" alt="Output of the previous code cell" />

In the figure, the lines and gray areas delimit the bounds; the outer-most (dash-dotted) lines delimit the quantum-bounds ($\pm 2$), whereas the inner (dashed) lines delimit the classical bounds ($\pm 2\sqrt{2}$). You can see that there are regions where the CHSH witness quantities exceeds the classical bounds. Congratulations! You have successfully demonstrated the violation of CHSH inequality in a real quantum system!

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3xxAgm1SF1wGp9k)